**What this notebook does:**
1. Cleans up the datasets for data visualization.

2. Concatinates categories together that are redunadant for more acurate comparison scores.

3. Creates an 'oportunity score' for gaging how good a city will be for a new company to locate in.


2. Creates a 'competition score', so you can compare a company to its same cateogries within the same geographical area.
3. Creates an 'overall z-score of category', so you can compare the company to its category across the state
4. Creats a 'city relative score' that is a z-score of the average rating in compairison to the area in which the company is located.
3. Creates a '95% confidence interval' for the probable average ratings a company could have.
4. Creates 2 datasets that can be used with these new columns of different sizes.

**Step 1:** clean datasets before merging and feature engineering, so Tableau is displayed without outliers and NULL data.

In [255]:
import numpy as np
from sklearn.neighbors import BallTree
import pandas as pd
import numpy as np
import gzip
import json

pd.set_option('display.max_columns', None)

def parse(path):
  g = open('Data/' + path, 'r')
  for l in g:
    yield json.loads(l)

def parse_first_n(path, n=10000):
    g = open('Data/' + path, 'r')
    for i, l in enumerate(g):
        if i >= n:
            break
        yield json.loads(l)

In [256]:
texas_metadata = pd.DataFrame(parse("meta-Texas.json"))
texas_reviews = pd.DataFrame(parse_first_n("review-Texas.json", n=1000000)) # is not cleaned yet
#add more or less reviews depending on how well tableu responds to the data.


In [257]:

category_reference_df = pd.read_csv('data/smaller_categories2.csv')
category_reference_df = category_reference_df.rename(columns = {'category':'small_group', 'subcategory': 'category_name'} )
category_reference_df

,industry,small_group,category_name
0,Agriculture & Natural Resources,Agricultural Services,agricultural
1,Agriculture & Natural Resources,Agricultural Services,agricultural association
2,Agriculture & Natural Resources,Agricultural Services,agricultural cooperative
3,Agriculture & Natural Resources,Agricultural Services,agricultural organization
4,Agriculture & Natural Resources,Agricultural Services,agricultural production
...,...,...,...
4269,Sports,Other Sports,table tennis facility
4270,Sports,Sports Facilities,american football field
4271,Sports,Sports Facilities,baseball field
4272,Sports,Sports Facilities,football field


In [258]:

# reshape a DataFrame from a wide format to a long format. => Convert the main category column to rows.
category_reference_df = category_reference_df.dropna()


In [259]:
# Clean the reference category 
category_reference_df['cateogry_name'] = (
    category_reference_df['category_name']
    .str.lower()
    .str.strip()
)



By using the .explode() method we are able to get more acuarate categories

like why would we compair a bridal shop and a boat dealer??

additonally for oportuity scores this would hurt business in specific categories for insights. 

Coaching service vs college

For 'Food & Dining', 'Hospitality & Tourism', 'Aviation'  think we defintely concatinate

but other than that we keep the .explode for calculation


larger categories as a reference they would give us more

In [260]:
#TODO do a left merge of the .exploded dataset to only concatinate categories specified above.

PLAN: Because we are grouping by category and city we just need to swap out the 3 columns above and keep the rest to make the data slighly better, we can add on an extra column for Tableau visualization but for our scores, no. 


PLAN: Include extra column but change the actual category for only specified using in method somehow. Then just use the rest of your code as it should work as is, but check for inconsitencies and the generalize the code to all 6 states.

**Clean the data**

In [261]:
texas_reviews = texas_reviews.dropna(subset=['rating']) # Drop rows where 'rating' is NaN

In [262]:
texas_metadata = texas_metadata.dropna(subset=['gmap_id', 'name']).drop_duplicates(subset=['gmap_id'], keep='first')



In [263]:
texas_metadata = texas_metadata[texas_metadata['state'] != 'Permanently closed']


In [264]:
texas_metadata = texas_metadata[(texas_metadata['latitude'] >= 25.5) & (texas_metadata['latitude'] <= 36.40)
                            & (texas_metadata['longitude'] >= -106.50) & (texas_metadata['longitude'] <= -93.31)]


In [265]:
texas_metadata.shape[0]

416224

**Add a city column to each company**

In [266]:
def standardize_rating_by_city(df):
    """
    Standardizes average ratings relative to each city's mean and standard deviation.
    
    Parameters:
    -----------
    df : pandas.DataFrame
        DataFrame with 'avg_rating' and 'city' columns
        
    Returns:
    --------
    pandas.Series
        Z-scores (standardized ratings) for each row
    """
    # Calculate city-level statistics
    city_stats = df.groupby('city')['avg_rating'].agg(['mean', 'std'])
    
    # Map city statistics to each row
    df_with_stats = df.copy()
    df_with_stats['city_mean'] = df['city'].map(city_stats['mean'])
    df_with_stats['city_std'] = df['city'].map(city_stats['std']) #map is like a mini merge
    
    # Calculate z-score: (value - mean) / std
    # Handle cases where std is 0 (all ratings in city are the same)
    standardized = (df_with_stats['avg_rating'] - df_with_stats['city_mean']) / df_with_stats['city_std'].replace(0, 1)
    
    return standardized

# Apply the function to add standardized ratings


In [267]:
path = 'Data/simplemaps_uscities_basicv1.93/uscities.csv'

us_cities = pd.read_csv(path)



# Convert to radians


category_df = texas_metadata.copy()
#THIS WILL FIRST ADD A CITY COLUMN TO OUR METADATA SO WE CAN GROUP
cities_rad = np.radians(us_cities[['lat', 'lng']])
companies_rad = np.radians(category_df[['latitude', 'longitude']])

# Build BallTree
tree = BallTree(cities_rad, metric='haversine')

# Find nearest city for each company
distances, indices = tree.query(companies_rad, k=1)

# Assign city name
nearest_city_names = us_cities.iloc[indices.flatten()]['city'].reset_index(drop=True)

category_df = category_df.reset_index(drop=True)
category_df['city'] = nearest_city_names
#now has a city column on category_df which we can use groupby to attatch a new column for
category_df['demand'] = category_df.groupby('city')['gmap_id'].transform('count')
df_for_later = category_df.copy()
category_df['standardized_city_rating'] = standardize_rating_by_city(category_df)
#make sure this column gets included in the final output
#Extract all unique category labels from california_metadata
category_df = category_df.explode('category')
#Convert everything to lower case and remove blank space 
category_df['category'] = category_df['category'].str.lower().str.strip()






**#TODO work with Mia to concatinate categories together in this cell's position**

In [268]:
# Left merge category_df with reference_long to map subcategories to main categories
category_df = category_df.merge(
    category_reference_df,
    left_on='category',
    right_on='category_name',
    how='left'
)






In [269]:
#TODO change main category NaN rows to 'Other' 
category_df[['industry', 'small_group']] = (
    category_df[['industry', 'small_group']].fillna('Other')
)

In [270]:
category_df

,name,address,gmap_id,description,latitude,longitude,category,avg_rating,num_of_reviews,price,hours,MISC,state,relative_results,url,city,demand,standardized_city_rating,industry,small_group,category_name,cateogry_name
0,Timewise Food Store,"Timewise Food Store, 1600 W Church St, Livings...",0x8638869e6b4e3529:0xe8d257447fe41672,None,30.713368,-94.954344,convenience store,4.8,4,None,"[[Thursday, Open 24 hours], [Friday, Open 24 h...","{'Service options': ['In-store shopping', 'Del...",Open 24 hours,"[0x863886bab3f9bb05:0x28a8062d0597dd34, 0x8638...",https://www.google.com/maps/place//data=!4m2!3...,Livingston,364,0.972932,Retail & Consumer Services,Grocery & Convenience,convenience store,convenience store
1,Goldstar Transit,"Goldstar Transit, 4415 W Dickson Ln, Little El...",0x864c3748dcc1c12d:0xbf904a61f262cf9b,None,33.159363,-96.975571,transportation service,4.5,4,None,"[[Thursday, 6AM–6PM], [Friday, 6AM–6PM], [Satu...",{'Accessibility': ['Wheelchair accessible entr...,Open ⋅ Closes 6PM,"[0x864c374855555555:0x3abb669a098bb235, 0x864c...",https://www.google.com/maps/place//data=!4m2!3...,Lakewood Village,14,0.243142,Professional Services,Other Professional,transportation service,transportation service
2,Walmart Pharmacy,"Walmart Pharmacy, 12220 FM 423, Frisco, TX 75033",0x864c3998b8d8dc83:0x57ffabe1e2322320,None,33.179867,-96.883691,pharmacy,3.3,24,$,"[[Thursday, 9AM–9PM], [Friday, 9AM–9PM], [Satu...","{'Service options': ['Curbside pickup', 'In-st...",Open ⋅ Closes 9PM,"[0x864c3999b29e291f:0x2d364c05e88eec13, 0x864c...",https://www.google.com/maps/place//data=!4m2!3...,Little Elm,161,-1.271692,Health & Medical Services,Pharmacy,pharmacy,pharmacy
3,Walmart Pharmacy,"Walmart Pharmacy, 12220 FM 423, Frisco, TX 75033",0x864c3998b8d8dc83:0x57ffabe1e2322320,None,33.179867,-96.883691,drug store,3.3,24,$,"[[Thursday, 9AM–9PM], [Friday, 9AM–9PM], [Satu...","{'Service options': ['Curbside pickup', 'In-st...",Open ⋅ Closes 9PM,"[0x864c3999b29e291f:0x2d364c05e88eec13, 0x864c...",https://www.google.com/maps/place//data=!4m2!3...,Little Elm,161,-1.271692,Retail & Consumer Services,Other Retail,drug store,drug store
4,Walmart Pharmacy,"Walmart Pharmacy, 12220 FM 423, Frisco, TX 75033",0x864c3998b8d8dc83:0x57ffabe1e2322320,None,33.179867,-96.883691,medical supply store,3.3,24,$,"[[Thursday, 9AM–9PM], [Friday, 9AM–9PM], [Satu...","{'Service options': ['Curbside pickup', 'In-st...",Open ⋅ Closes 9PM,"[0x864c3999b29e291f:0x2d364c05e88eec13, 0x864c...",https://www.google.com/maps/place//data=!4m2!3...,Little Elm,161,-1.271692,Retail & Consumer Services,Other Retail,medical supply store,medical supply store
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
973846,O'Reilly Auto Parts,"O'Reilly Auto Parts, 702 Tennessee Ave, Dalhar...",0x8705b6b8981c266d:0x5770fa9f1b6c0b1a,None,36.058740,-102.514025,car repair and maintenance,4.3,94,None,"[[Wednesday, 7AM–9PM], [Thursday, 7AM–9PM], [F...","{'Service options': ['Curbside pickup', 'In-st...",NaN,"[0x8705b6c7859a9041:0x9107ec50aa4d5609, 0x8705...",https://www.google.com/maps/place//data=!4m2!3...,Dalhart,150,0.181768,Automotive & Transportation,Auto Repair,car repair and maintenance,car repair and maintenance
973847,O'Reilly Auto Parts,"O'Reilly Auto Parts, 702 Tennessee Ave, Dalhar...",0x8705b6b8981c266d:0x5770fa9f1b6c0b1a,None,36.058740,-102.514025,car stereo store,4.3,94,None,"[[Wednesday, 7AM–9PM], [Thursday, 7AM–9PM], [F...","{'Service options': ['Curbside pickup', 'In-st...",NaN,"[0x8705b6c7859a9041:0x9107ec50aa4d5609, 0x8705...",https://www.google.com/maps/place//data=!4m2!3...,Dalhart,150,0.181768,Retail & Consumer Services,Other Retail,car stereo store,car stereo store
973848,O'Reilly Auto Parts,"O'Reilly Auto Parts, 702 Tennessee Ave, Dalhar...",0x8705b6b8981c266d:0x5770fa9f1b6c0b1a,None,36.058740,-102.514025,motorcycle parts store,4.3,94,None,"[[Wednesday, 7AM–9PM], [Thursday, 7AM–9PM], [F...","{'Service options': ['Curbside pickup', 'In-st...",NaN,

In [271]:
match_rate = category_df['small_group'].notna().mean() * 100
print(f"Match rate: {match_rate:.2f}%")
category_df[category_df['small_group'].isna()]

Match rate: 100.00%


,name,address,gmap_id,description,latitude,longitude,category,avg_rating,num_of_reviews,price,hours,MISC,state,relative_results,url,city,demand,standardized_city_rating,industry,small_group,category_name,cateogry_name


In [272]:
category_df.drop(columns=['cateogry_name'], inplace=True)
#TODO make a main competition score (same), and a general competition score which will more broadly approach the city's competitive landscape.
#OTHER COMPETITION SCORE WILL INSTEAD JUST GROUP BY SMALL_GROUP instead of the catgeogry column and make new column
#TODO make sure all code is bullet proof so that we can begin to export csvs and start generalizing the code to other states.
#OPORTUNITY score needs to remain as is so that we can track niche oportunities.

In [273]:
category_stats = category_df.groupby('category')['avg_rating'].agg(['mean', 'std'])
small_category_stats = category_df.groupby('small_group')['avg_rating'].agg(['mean', 'std'])
#rn category stats is just grouped by category as a whole
# Map city statistics to each row
df_with_stats = category_df.copy()

df_with_stats['cat_mean'] = df_with_stats['category'].map(category_stats['mean'])
df_with_stats['cat_std'] = df_with_stats['category'].map(category_stats['std'])
df_with_stats['small_cat_mean'] = df_with_stats['small_group'].map(small_category_stats['mean'])
df_with_stats['small_cat_std'] = df_with_stats['small_group'].map(small_category_stats['std'])

# Calculate z-score: (value - mean) / std
# Handle cases where std is 0 (all ratings in city are the same)
category_df['Entire_category_score'] = (df_with_stats['avg_rating'] - df_with_stats['cat_mean']) / df_with_stats['cat_std'].replace(0, 1)
#this remains the same as the entire category is large enough to use niche categories


In [274]:
# # # #Remove tiny categories 
category_counts= category_df.get('category').value_counts()
small_category_counts = category_df.get('small_group').value_counts()


# Setting threshold for the category 
category_counts.describe()
small_category_counts.describe()
category_counts_threshold = 35 #median, we could alter this if there are too many categories for our deomo

#this only necessary for category_counts categories as they are much more niche

# Map category_count to each category in the df
category_df['category_count'] = category_df['category'].map(category_counts)
category_df['small_category_count'] = category_df['small_group'].map(small_category_counts)
# category_df = category_df[category_df.get('category_count') >= category_counts_threshold]
# category_df = category_df.sort_values(by='category_count', ascending=False)


**Aggregate to each category within each city**

In [275]:

# Calculate mean, std, and count for each category within each city
category_stats = category_df.groupby(["city", "category"])['avg_rating'].agg(['mean', 'std', 'count']).reset_index()
category_stats.columns = ['city', 'category', 'cc_mean', 'cc_std', 'cc_count'] 

small_category_stats = category_df.groupby(["city", "small_group"])['avg_rating'].agg(['mean', 'std', 'count']).reset_index()
small_category_stats.columns = ['scity', 'scategory', 'scc_mean', 'scc_std', 'scc_count'] 

 # rename columns for clarity
#use these for our average rating of category in the city column

In [276]:
category_stats
small_category_stats

,scity,scategory,scc_mean,scc_std,scc_count
0,Abbott,Church,5.000000,0.000000,2
1,Abbott,Clothing & Accessories,4.900000,NaN,1
2,Abbott,Consumer Services,4.250000,0.494975,2
3,Abbott,Other,4.700000,0.244949,8
4,Abbott,Other Retail,4.533333,0.404145,3
...,...,...,...,...,...
48527,Zuehl,Other Industrial,4.350000,0.876682,8
48528,Zuehl,Other Professional,4.240000,0.650385,5
48529,Zuehl,Other Public Services,3.900000,NaN,1
48530,Zuehl,Other Retail,3.937500,0.720986,8


In [277]:

# Merge back into category data set (here category df is just the exploded metadata with city and category count)


mcategory_df = category_df.merge(category_stats, on=['city', 'category'], how='inner').merge(small_category_stats, left_on = ['city', 'small_group'], right_on = ['scity', 'scategory'], how= 'inner') #merge on both city and category
mcategory_df['cc_count'] = (
    mcategory_df.groupby(['city', 'category'])['gmap_id']
      .transform('nunique')
)
###########
#this does not need to change as we want oportuity score to remain niche
###########

mcategory_df
mcategory_df.drop(columns=[ 'category_count'])



#s should be greater for count and std should be lower


,name,address,gmap_id,description,latitude,longitude,category,avg_rating,num_of_reviews,price,hours,MISC,state,relative_results,url,city,demand,standardized_city_rating,industry,small_group,category_name,Entire_category_score,small_category_count,cc_mean,cc_std,cc_count,scity,scategory,scc_mean,scc_std,scc_count
0,Timewise Food Store,"Timewise Food Store, 1600 W Church St, Livings...",0x8638869e6b4e3529:0xe8d257447fe41672,None,30.713368,-94.954344,convenience store,4.8,4,None,"[[Thursday, Open 24 hours], [Friday, Open 24 h...","{'Service options': ['In-store shopping', 'Del...",Open 24 hours,"[0x863886bab3f9bb05:0x28a8062d0597dd34, 0x8638...",https://www.google.com/maps/place//data=!4m2!3...,Livingston,364,0.972932,Retail & Consumer Services,Grocery & Convenience,convenience store,1.587334,21904,3.85,0.559876,14,Livingston,Grocery & Convenience,4.011538,0.464178,26
1,Goldstar Transit,"Goldstar Transit, 4415 W Dickson Ln, Little El...",0x864c3748dcc1c12d:0xbf904a61f262cf9b,None,33.159363,-96.975571,transportation service,4.5,4,None,"[[Thursday, 6AM–6PM], [Friday, 6AM–6PM], [Satu...",{'Accessibility': ['Wheelchair accessible entr...,Open ⋅ Closes 6PM,"[0x864c374855555555:0x3abb669a098bb235, 0x864c...",https://www.google.com/maps/place//data=!4m2!3...,Lakewood Village,14,0.243142,Professional Services,Other Professional,transportation service,0.702763,108201,4.50,NaN,1,Lakewood Village,Other Professional,4.220000,0.778603,10
2,Walmart Pharmacy,"Walmart Pharmacy, 12220 FM 423, Frisco, TX 75033",0x864c3998b8d8dc83:0x57ffabe1e2322320,None,33.179867,-96.883691,pharmacy,3.3,24,$,"[[Thursday, 9AM–9PM], [Friday, 9AM–9PM], [Satu...","{'Service options': ['Curbside pickup', 'In-st...",Open ⋅ Closes 9PM,"[0x864c3999b29e291f:0x2d364c05e88eec13, 0x864c...",https://www.google.com/maps/place//data=!4m2!3...,Little Elm,161,-1.271692,Health & Medical Services,Pharmacy,pharmacy,-0.733855,2133,3.15,0.212132,2,Little Elm,Pharmacy,3.150000,0.212132,2
3,Walmart Pharmacy,"Walmart Pharmacy, 12220 FM 423, Frisco, TX 75033",0x864c3998b8d8dc83:0x57ffabe1e2322320,None,33.179867,-96.883691,drug store,3.3,24,$,"[[Thursday, 9AM–9PM], [Friday, 9AM–9PM], [Satu...","{'Service options': ['Curbside pickup', 'In-st...",Open ⋅ Closes 9PM,"[0x864c3999b29e291f:0x2d364c05e88eec13, 0x864c...",https://www.google.com/maps/place//data=!4m2!3...,Little Elm,161,-1.271692,Retail & Consumer Services,Other Retail,drug store,-0.443480,182960,2.50,0.721110,3,Little Elm,Other Retail,4.086735,0.643867,98
4,Walmart Pharmacy,"Walmart Pharmacy, 12220 FM 423, Frisco, TX 75033",0x864c3998b8d8dc83:0x57ffabe1e2322320,None,33.179867,-96.883691,medical supply store,3.3,24,$,"[[Thursday, 9AM–9PM], [Friday, 9AM–9PM], [Satu...","{'Service options': ['Curbside pickup', 'In-st...",Open ⋅ Closes 9PM,"[0x864c3999b29e291f:0x2d364c05e88eec13, 0x864c...",https://www.google.com/maps/place//data=!4m2!3...,Little Elm,161,-1.271692,Retail & Consumer Services,Other Retail,medical supply store,-0.378974,182960,3.30,NaN,1,Little Elm,Other Retail,4.086735,0.643867,98
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
972272,O'Reilly Auto Parts,"O'Reilly Auto Parts, 702 Tennessee Ave, Dalhar...",0x8705b6b8981c266d:0x5770fa9f1b6c0b1a,None,36.058740,-102.514025,car repair and maintenance,4.3,94,None,"[[Wednesday, 7AM–9PM], [Thursday, 7AM–9PM], [F...","{'Service options': ['Curbside pickup', 'In-st...",NaN,"[0x8705b6c7859a9041:0x9107ec50aa4d5609, 0x8705...",https://www.google.com/maps/place//data=!4m2!3...,Dalhart,150,0.181768,Automotive & Transportation,Auto Repair,car repair and maintenance,-0.189205,3436,4.30,NaN,1,Dalhart,Auto Repair,4.300000,NaN,1
972273,O'Reilly Auto Parts,"O'Reilly Auto Parts, 702 Tennessee Ave, Dalhar...",0x8705b6b8981c266d:0x5770fa9f1b6c0b1a,None,36.058740,-102.514025,car stereo store,4.3,94,None,"[[Wednesday, 7AM–9PM], [Thursday, 7AM–9PM], [F...","{'Service options': ['Curbside pickup', 'In-st...",NaN,"[0x8705

In [278]:
mcategory_df.columns #lets use cc_mean and keep it as a column 
#so we can do 5 - df['cc_mean'] to get our quality penalty
#TODO check if this includes the 'demand' column and the cc_mean

Index(['name', 'address', 'gmap_id', 'description', 'latitude', 'longitude',
       'category', 'avg_rating', 'num_of_reviews', 'price', 'hours', 'MISC',
       'state', 'relative_results', 'url', 'city', 'demand',
       'standardized_city_rating', 'industry', 'small_group', 'category_name',
       'Entire_category_score', 'category_count', 'small_category_count',
       'cc_mean', 'cc_std', 'cc_count', 'scity', 'scategory', 'scc_mean',
       'scc_std', 'scc_count'],
      dtype='object')

In [279]:

# competition score = normalized rating in local area relative to competition
# If less than 2 companies in the city-category combo, mark as insufficient data
mcategory_df['competition_score'] = mcategory_df.apply(
    lambda row: 
        (row['avg_rating'] - row['cc_mean']) / row['cc_std']
        if (row['cc_count'] >= 2 and row['cc_std'] != 0)
        else np.nan,
    axis=1
)

mcategory_df['broad_competition_score'] = mcategory_df.apply(
    lambda row: 
        (row['avg_rating'] - row['scc_mean']) / row['scc_std']
        if (row['scc_count'] >= 2 and row['scc_std'] != 0)
        else np.nan,
    axis=1
)



mcategory_df = mcategory_df.groupby('gmap_id').agg({
    'competition_score': 'mean',
    'broad_competition_score': 'mean',
    'standardized_city_rating': 'mean',
    'Entire_category_score': 'mean',
    'cc_mean': 'mean',
    'cc_count': 'mean', #these two are for oportunity score later on
    'demand': 'first'
    
    #this gives us cc_mean as a col to use 'Targetability'
}).reset_index()

mcategory_df #The reason that the number dropped is becuase explode will create multiple rows and grouping by gmap_id will reduce that back


,gmap_id,competition_score,broad_competition_score,standardized_city_rating,Entire_category_score,cc_mean,cc_count,demand
0,0x0:0x3de3e25d19fc7da6,-0.142294,-0.657796,-0.708316,-0.639810,3.764031,8.714286,1589
1,0x0:0x48c466d0f3de5f5a,0.468495,0.574539,0.879989,0.514483,4.775697,31.200000,3164
2,0x0:0x554de8957fed30a1,0.116645,0.000535,-0.108407,0.196436,4.077594,49.400000,1819
3,0x0:0x5702eb6d2d1301db,1.333021,1.329392,1.353825,1.148322,4.131818,22.000000,916
4,0x1000000000000001:0xe85d5b0e25befcb7,1.292591,0.797106,0.552496,0.460215,3.850000,4.000000,1362
...,...,...,...,...,...,...,...,...
414645,0xaceec0ea6c38bcfd:0xf291bc79aa31fa60,0.707107,0.952983,1.233789,0.692342,4.350000,2.000000,212
414646,0xad33c203e03d89f9:0xb8ee8c778a581ca4,0.519618,0.991805,1.140207,0.707797,4.839474,10.500000,1492
414647,0xd0c7e1a240869ad:0xafa498adc8911c71,0.712855,0.877136,1.011768,0.736033,4.425439,106.000000,7526
414648,0xdce6e3016b65401:0xf220f369359026f2,1.421060,1.037311,1.102625,1.339574,4.195769,35.000000,2889


In [280]:
#We will merge all of this back into texas metadata with an inner merge to add the new z-score and CI columns for the Tableu dataset.

company_stats = (
    texas_reviews #first time using texas reviews, we should definetly do a left merge for CI, since limmited data n=1000000
        .groupby('gmap_id')['rating']
        .agg(['mean', 'std', 'count'])
        .reset_index()
)

def compute_ci(row):
    mean = row['mean']
    std = row['std']
    n = row['count']
    
    if n == 0:
        return None
    
    margin = 1.96 * (std / np.sqrt(n))
    return (mean - margin, mean + margin)

company_stats['CI'] = company_stats.apply(compute_ci, axis=1)

mtexas_metadata = texas_metadata.merge(
    company_stats[['gmap_id', 'CI']],
    on='gmap_id',
    how='inner'
)

# mcategory_df['CI']= mcategory_df['gmap_id'].apply(individual_company_rating_confidence_interval)
# mcategory_df #THIS TAKES TO LONG SPEED IT UP AND THEN MERGE BACK TO TEXAS METADATA



The issue is that out output code has a list of categories in the category column, we don't want only one category using the .explode method becasue that would duplicate out gmap_ids. This would cause our data visualization to crash.

**<u>The list format of categories doesen't matter becuse the competition score already finds the competition relative to all of the companies categories, This score is the most important as it is honestly more acurate at it takes into account multiple varibales rather than just one, calculates them separately and then avereges the scores.**



In [281]:


final_texas_metadata = texas_metadata.merge(mtexas_metadata[['gmap_id', 'CI']], on='gmap_id', how='left').merge(mcategory_df[['gmap_id', 'competition_score','broad_competition_score', 'standardized_city_rating', 'Entire_category_score', 'cc_count', 'demand', 'cc_mean']], on='gmap_id', how='left')
final_texas_metadata['competition_score'] = final_texas_metadata['competition_score'].fillna("not enough data")
final_texas_metadata['broad_competition_score'] = final_texas_metadata['broad_competition_score'].fillna("not enough data")
final_texas_metadata = final_texas_metadata.merge(df_for_later[['gmap_id','city']], on= 'gmap_id') #WTF IS WRONG
#shuold be > 409k
final_texas_metadata['category'] = final_texas_metadata['category'].apply(lambda x: tuple(x) if x is not None else None)

****BELOW ARE 2 FINAL DATASETS. ONE WITH ONLY THE COMPANIES IN TEXAS REVIEWS WITH NO NAN VALUES this is small_... AND THE OTHER INCLUDES ALL THE DATA IN META BUT NAN VALUES. this is big_.... Choose which one to use BASED ON HOW WELL TABLEAU HANDLES A LOT OF DATA.****

In [282]:
small_test_final_texas_metadata = final_texas_metadata.dropna(subset=['CI'])
small_test_final_texas_metadata = small_test_final_texas_metadata[~small_test_final_texas_metadata['CI'].apply(lambda x: any(pd.isna(i) for i in x))]
small_test_final_texas_metadata['CI'] = small_test_final_texas_metadata['CI'].apply(lambda x: (x[0],5.0) if x[1] > 5.0 else (1.0, x[1]) if x[0] < 1.0 else x)
small_test_final_texas_metadata = small_test_final_texas_metadata.rename(columns={
    'CI': 'CI_for_avg_rating',
    'Entire_category_score': 'category_relative_score'
})

# Merge in city and category from category_df (which has the city/category mapping)






**Adjusting confidence intervals**

In [283]:
big_test_final_texas_metadata = final_texas_metadata.copy()

def clamp_ci(x): #this fucniton is only needed for big due to NaN handling
    # Skip NaN or non-tuples
    if not isinstance(x, (tuple, list)):
        return x
    
    lower, upper = x
    
    # Clamp values
    lower = max(1.0, lower)
    upper = min(5.0, upper)
    
    return (lower, upper)

big_test_final_texas_metadata['CI'] = (
    big_test_final_texas_metadata['CI'].apply(clamp_ci)
)

# Rename columns
big_test_final_texas_metadata = big_test_final_texas_metadata.rename(columns={
    'CI': 'CI_for_avg_rating',
    'Entire_category_score': 'category_relative_score'
})


In [284]:
print(big_test_final_texas_metadata.shape[0], small_test_final_texas_metadata.shape[0]) #big vs small difference

416224 34376


In [285]:
# Calculate opportunity_score per city-category combination
def calculate_opportunity_score(df):
    """
    Calculate opportunity score per city-category combination.
    
    Formula: ((5 - cc_mean) * demand) / cc_count
    """
    # Use the category column (handle case where there might be duplicates)

    
    opportunity_scores = df.groupby(['city', 'category']).apply( #category isnt hashable 
        lambda group: ((5 - group['cc_mean'].iloc[0]) * group['demand'].iloc[0]) / group['cc_count'].iloc[0],
        include_groups=False
    ).reset_index(name='opportunity_score')
    opportunity_scores.columns = ['city', 'category', 'opportunity_score']

    # Merge back to preserve all rows
    result = df.merge(
        opportunity_scores,
        on=['city', 'category'],
        how='left'
    )
    return result

small_test_final_texas_metadata = calculate_opportunity_score(small_test_final_texas_metadata)
big_test_final_texas_metadata = calculate_opportunity_score(big_test_final_texas_metadata)

In [286]:
small_test_final_texas_metadata.isna().sum()
#null valus simply stem from the NaN in category so no need to change as ['name', 'gmap', 'avg_rating'] all have 0 NaNs.

name                            0
address                      1366
gmap_id                         0
description                 32674
latitude                        0
longitude                       0
category                      178
avg_rating                      0
num_of_reviews                  0
price                       32723
hours                        8532
MISC                         8109
state                        8053
relative_results             4408
url                             0
CI_for_avg_rating               0
competition_score               0
broad_competition_score         0
standardized_city_rating      188
category_relative_score       185
cc_count                      178
demand                        178
cc_mean                       178
city                            0
opportunity_score             178
dtype: int64

**try and add this funciton above**

In [287]:
big_test_final_texas_metadata.isna().sum()
#null valus simply stem from the NaN in category so no need to change as ['name', 'gmap', 'avg_rating'] all have 0 NaNs.

name                             0
address                       8551
gmap_id                          0
description                 340106
latitude                         0
longitude                        0
category                      1574
avg_rating                       0
num_of_reviews                   0
price                       340669
hours                        85151
MISC                         67655
state                       117313
relative_results             37575
url                              0
CI_for_avg_rating           379338
competition_score                0
broad_competition_score          0
standardized_city_rating      1640
category_relative_score       1661
cc_count                      1574
demand                        1574
cc_mean                       1574
city                             0
opportunity_score             1574
dtype: int64

In [288]:
small_test_final_texas_metadata
check = (
    small_test_final_texas_metadata.groupby(['city', 'category'])
      .agg({
          'cc_mean': 'nunique',
          'demand': 'nunique',
          'cc_count': 'nunique'
      })
)

check.head()

problem_groups = check[
    (check['cc_mean'] > 1) |
    (check['demand'] > 1) |
    (check['cc_count'] > 1)
]

problem_groups
#No duplicates AGTG

,,cc_mean,demand,cc_count
city,category,,,


In [289]:

check2 = (
    big_test_final_texas_metadata.groupby(['city', 'category'])
      .agg({
          'cc_mean': 'nunique',
          'demand': 'nunique',
          'cc_count': 'nunique'
      })
)

check2.head()

problem_groups = check2[
    (check2['cc_mean'] > 1) |
    (check2['demand'] > 1) |
    (check2['cc_count'] > 1)
]

problem_groups
#none so we good 

,,cc_mean,demand,cc_count
city,category,,,


***FINAL DATA FRAMES!!!!!***

In [ ]:
# Add small_group column from category_df
small_test_final_texas_metadata = small_test_final_texas_metadata.merge(
    category_df[['gmap_id', 'small_group']].drop_duplicates(),
    on='gmap_id',
    how='left'
)

big_test_final_texas_metadata = big_test_final_texas_metadata.merge(
    category_df[['gmap_id', 'small_group']].drop_duplicates(),
    on='gmap_id',
    how='left'
)

# Columns to drop
columns_to_drop = ['description', 'price', 'hours', 'MISC', 'state', 'relative_results', 'url', 'cc_count', 'demand', 'cc_mean']

# Drop unwanted columns
small_test_final_texas_metadata = small_test_final_texas_metadata.drop(
    columns=[col for col in columns_to_drop if col in small_test_final_texas_metadata.columns]
)

big_test_final_texas_metadata = big_test_final_texas_metadata.drop(
    columns=[col for col in columns_to_drop if col in big_test_final_texas_metadata.columns]
)

# Reorder columns: put city right next to gmap_id
def reorder_columns(df):
    cols = df.columns.tolist()
    # Remove city from its current position if it exists
    if 'city' in cols:
        cols.remove('city')
    if 'small_group' in cols:
        cols.remove('small_group')
    # Insert city right after gmap_id
    if 'gmap_id' in cols:
        gmap_idx = cols.index('gmap_id')
        cols.insert(gmap_idx + 1, 'city')
    if 'category' in cols:
        category_idx = cols.index('category')
        cols.insert(category_idx + 1, 'small_group') 

    
    return df[cols]
    

small_test_final_texas_metadata = reorder_columns(small_test_final_texas_metadata)
big_test_final_texas_metadata = reorder_columns(big_test_final_texas_metadata)

# Rename small_group to broad_category

#TODO Columns that we dont need: description, price, hours, misc, state, relative_reulsts, url, cc_count, demand, cc_mean
#TODO Columns that we want to add: the small_goups for data viz and reorder the city columnto be right next to the gmap_id in terms of where the columns are ordered.

,name,address,gmap_id,city,latitude,longitude,category,small_group,avg_rating,num_of_reviews,CI_for_avg_rating,competition_score,broad_competition_score,standardized_city_rating,category_relative_score,opportunity_score
0,Timewise Food Store,"Timewise Food Store, 1600 W Church St, Livings...",0x8638869e6b4e3529:0xe8d257447fe41672,Livingston,30.713368,-94.954344,"(Convenience store,)",Grocery & Convenience,4.8,4,"(4.429219701353091, 5.0)",1.696803,1.698618,0.972932,1.587334,29.900000
1,Goldstar Transit,"Goldstar Transit, 4415 W Dickson Ln, Little El...",0x864c3748dcc1c12d:0xbf904a61f262cf9b,Lakewood Village,33.159363,-96.975571,"(Transportation service,)",Other Professional,4.5,4,"(3.858439402706183, 5.0)",not enough data,0.359619,0.243142,0.702763,7.000000
2,Walmart Pharmacy,"Walmart Pharmacy, 12220 FM 423, Frisco, TX 75033",0x864c3998b8d8dc83:0x57ffabe1e2322320,Little Elm,33.179867,-96.883691,"(Pharmacy, Drug store, Medical supply store, V...",Pharmacy,3.3,24,"(2.773104882616715, 3.810228450716618)",0.252294,-0.739641,-1.271692,-0.716826,124.327778
3,Walmart Pharmacy,"Walmart Pharmacy, 12220 FM 423, Frisco, TX 75033",0x864c3998b8d8dc83:0x57ffabe1e2322320,Little Elm,33.179867,-96.883691,"(Pharmacy, Drug store, Medical supply store, V...",Other Retail,3.3,24,"(2.773104882616715, 3.810228450716618)",0.252294,-0.739641,-1.271692,-0.716826,124.327778
4,Luminous Logistics,"Luminous Logistics, 3838 W Miller Rd, Garland,...",0x864ea0993bffffff:0xb50b5bb2fccf9d9b,Garland,32.893678,-96.688611,"(Delivery service,)",Other Professional,2.3,8,"(1.382640020906352, 3.117359979093648)",not enough data,-2.766581,-2.768392,-3.818610,8208.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
664787,Pilot Travel Center,"Pilot Travel Center, 100 S Poplar St, Stratfor...",0x8705dff80d5a8531:0xcfd4e9fa1f92f376,Stratford,36.331260,-102.073430,"(Truck stop, Convenience store, Gas station)",Other,3.9,1593,NaN,-0.259585,-0.312014,-0.449284,0.193181,15.858333
664788,Pilot Travel Center,"Pilot Travel Center, 100 S Poplar St, Stratfor...",0x8705dff80d5a8531:0xcfd4e9fa1f92f376,Stratford,36.331260,-102.073430,"(Truck stop, Convenience store, Gas station)",Grocery & Convenience,3.9,1593,NaN,-0.259585,-0.312014,-0.449284,0.193181,15.858333
664789,O'Reilly Auto Parts,"O'Reilly Auto Parts, 702 Tennessee Ave, Dalhar...",0x8705b6b8981c266d:0x5770fa9f1b6c0b1a,Dalhart,36.058740,-102.514025,"(Auto parts store, Auto body parts supplier, C...",Other Retail,4.3,94,NaN,-0.707107,-0.146063,0.181768,-0.039905,74.318182
664790,O'Reilly Auto Parts,"O'Reilly Auto Parts, 702 Tennessee Ave, Dalhar...",0x8705b6b8981c266d:0x5770fa9f1b6c0b1a,Dalhart,36.058740,-102.514025,"(Auto parts store, Auto body parts supplier, C...",Auto Body,4.3,94,NaN,-0.707107,-0.146063,0.181768,-0.039905,74.318182


In [ ]:
small_test_final_texas_metadata = small_test_final_texas_metadata.rename(columns={'small_group': 'broad_category'})
big_test_final_texas_metadata = big_test_final_texas_metadata.rename(columns={'small_group': 'broad_category'})

big_test_final_texas_metadata

,name,address,gmap_id,city,latitude,longitude,category,broad_category,avg_rating,num_of_reviews,CI_for_avg_rating,competition_score,broad_competition_score,standardized_city_rating,category_relative_score,opportunity_score
0,Timewise Food Store,"Timewise Food Store, 1600 W Church St, Livings...",0x8638869e6b4e3529:0xe8d257447fe41672,Livingston,30.713368,-94.954344,"(Convenience store,)",Grocery & Convenience,4.8,4,"(4.429219701353091, 5.0)",1.696803,1.698618,0.972932,1.587334,29.900000
1,Goldstar Transit,"Goldstar Transit, 4415 W Dickson Ln, Little El...",0x864c3748dcc1c12d:0xbf904a61f262cf9b,Lakewood Village,33.159363,-96.975571,"(Transportation service,)",Other Professional,4.5,4,"(3.858439402706183, 5.0)",not enough data,0.359619,0.243142,0.702763,7.000000
2,Walmart Pharmacy,"Walmart Pharmacy, 12220 FM 423, Frisco, TX 75033",0x864c3998b8d8dc83:0x57ffabe1e2322320,Little Elm,33.179867,-96.883691,"(Pharmacy, Drug store, Medical supply store, V...",Pharmacy,3.3,24,"(2.773104882616715, 3.810228450716618)",0.252294,-0.739641,-1.271692,-0.716826,124.327778
3,Walmart Pharmacy,"Walmart Pharmacy, 12220 FM 423, Frisco, TX 75033",0x864c3998b8d8dc83:0x57ffabe1e2322320,Little Elm,33.179867,-96.883691,"(Pharmacy, Drug store, Medical supply store, V...",Other Retail,3.3,24,"(2.773104882616715, 3.810228450716618)",0.252294,-0.739641,-1.271692,-0.716826,124.327778
4,Luminous Logistics,"Luminous Logistics, 3838 W Miller Rd, Garland,...",0x864ea0993bffffff:0xb50b5bb2fccf9d9b,Garland,32.893678,-96.688611,"(Delivery service,)",Other Professional,2.3,8,"(1.382640020906352, 3.117359979093648)",not enough data,-2.766581,-2.768392,-3.818610,8208.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
664787,Pilot Travel Center,"Pilot Travel Center, 100 S Poplar St, Stratfor...",0x8705dff80d5a8531:0xcfd4e9fa1f92f376,Stratford,36.331260,-102.073430,"(Truck stop, Convenience store, Gas station)",Other,3.9,1593,NaN,-0.259585,-0.312014,-0.449284,0.193181,15.858333
664788,Pilot Travel Center,"Pilot Travel Center, 100 S Poplar St, Stratfor...",0x8705dff80d5a8531:0xcfd4e9fa1f92f376,Stratford,36.331260,-102.073430,"(Truck stop, Convenience store, Gas station)",Grocery & Convenience,3.9,1593,NaN,-0.259585,-0.312014,-0.449284,0.193181,15.858333
664789,O'Reilly Auto Parts,"O'Reilly Auto Parts, 702 Tennessee Ave, Dalhar...",0x8705b6b8981c266d:0x5770fa9f1b6c0b1a,Dalhart,36.058740,-102.514025,"(Auto parts store, Auto body parts supplier, C...",Other Retail,4.3,94,NaN,-0.707107,-0.146063,0.181768,-0.039905,74.318182
664790,O'Reilly Auto Parts,"O'Reilly Auto Parts, 702 Tennessee Ave, Dalhar...",0x8705b6b8981c266d:0x5770fa9f1b6c0b1a,Dalhart,36.058740,-102.514025,"(Auto parts store, Auto body parts supplier, C...",Auto Body,4.3,94,NaN,-0.707107,-0.146063,0.181768,-0.039905,74.318182


In [291]:
small_test_final_texas_metadata.head()

,name,address,gmap_id,city,latitude,longitude,category,small_group,avg_rating,num_of_reviews,CI_for_avg_rating,competition_score,broad_competition_score,standardized_city_rating,category_relative_score,opportunity_score
0,Timewise Food Store,"Timewise Food Store, 1600 W Church St, Livings...",0x8638869e6b4e3529:0xe8d257447fe41672,Livingston,30.713368,-94.954344,"(Convenience store,)",Grocery & Convenience,4.8,4,"(4.429219701353091, 5.0)",1.696803,1.698618,0.972932,1.587334,29.900000
1,Goldstar Transit,"Goldstar Transit, 4415 W Dickson Ln, Little El...",0x864c3748dcc1c12d:0xbf904a61f262cf9b,Lakewood Village,33.159363,-96.975571,"(Transportation service,)",Other Professional,4.5,4,"(3.858439402706183, 5.0)",not enough data,0.359619,0.243142,0.702763,7.000000
2,Walmart Pharmacy,"Walmart Pharmacy, 12220 FM 423, Frisco, TX 75033",0x864c3998b8d8dc83:0x57ffabe1e2322320,Little Elm,33.179867,-96.883691,"(Pharmacy, Drug store, Medical supply store, V...",Pharmacy,3.3,24,"(2.773104882616715, 3.810228450716618)",0.252294,-0.739641,-1.271692,-0.716826,124.327778
3,Walmart Pharmacy,"Walmart Pharmacy, 12220 FM 423, Frisco, TX 75033",0x864c3998b8d8dc83:0x57ffabe1e2322320,Little Elm,33.179867,-96.883691,"(Pharmacy, Drug store, Medical supply store, V...",Other Retail,3.3,24,"(2.773104882616715, 3.810228450716618)",0.252294,-0.739641,-1.271692,-0.716826,124.327778
4,Luminous Logistics,"Luminous Logistics, 3838 W Miller Rd, Garland,...",0x864ea0993bffffff:0xb50b5bb2fccf9d9b,Garland,32.893678,-96.688611,"(Delivery service,)",Other Professional,2.3,8,"(1.382640020906352, 3.117359979093648)",not enough data,-2.766581,-2.768392,-3.818610,8208.000000


In [294]:
small_test_final_texas_metadata['state'] = 'Texas'
big_test_final_texas_metadata['state'] = 'Texas'

Finally download the datasets

In [ ]:
# big_test_final_texas_metadata.to_csv('Texas_LL_Big.csv', index=False)
# small_test_final_texas_metadata.to_csv('Texas_LL_Small.csv', index=False)